[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/w05-w06-pca/lab/lab05_pca_theory.ipynb)

# Lab 05 — PCA I: from theory to code

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
np.set_printoptions(precision=4, suppress=True)

**Course:** 040563107 Tools for Statistical Data Science · **Week 5** · **Time:** 3 hours · **Graded:** lab rubric (/8)

**After this lab you can:**
* compute PCA by hand for a 2×2 covariance matrix and check it in code (objective PCA-O2);
* implement PCA through the SVD of the centred data, and verify $\lambda_j=\sigma_j^2/(n-1)$ and the reconstruction-error identity (PCA-O3);
* choose between covariance and correlation PCA and interpret loadings, allowing for sign indeterminacy (PCA-O4);
* diagnose three PCA failure modes: uncentred data, `eig` vs `eigh`, and sign flips (PCA-O8).

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes.
3. Write the answers marked **Q**.
4. Before submitting, use *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter). The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

Self-checks test *consistency*, not your interpretation. Your written answers carry half of the marks.

## Part 1 — The hand example, checked in code (about 35 minutes)

These are the five observations from lecture worked example A. Do the computation **on paper first**, then confirm it here.

In [ ]:
X_hand = np.array([[6, 6],
                   [2, 5],
                   [0, 2],
                   [0, 6],
                   [2, 6]], dtype=float)
n_hand = X_hand.shape[0]
X_hand

In [ ]:
# Task T1.1
# Compute the mean vector `xbar`, the centred data `Xc`, and the sample covariance `S`
# using the matrix formula S = Xc^T Xc / (n - 1). Do not use np.cov here.
# YOUR CODE HERE
raise NotImplementedError()
print("mean:", xbar)
print("S =\n", S)

In [ ]:
# Self-check for task T1.1: run it; it should print OK.
assert np.allclose(S, np.cov(X_hand, rowvar=False)), "S does not match np.cov: check the centring and the divisor"
assert np.allclose(Xc.mean(axis=0), 0), "Xc must have column means equal to zero"
print("OK")

In [ ]:
# Task T1.2
# Use np.linalg.eigh (not eig) on S. eigh returns eigenvalues in ASCENDING order:
# reorder so that `lam` is DESCENDING and the columns of `V` match.
# Then compute the scores `Z` (n x 2) and the explained-variance ratios `evr`.
# YOUR CODE HERE
raise NotImplementedError()
print("eigenvalues:", lam)
print("eigenvectors (columns):\n", V)
print("scores PC1 x sqrt(5):", Z[:, 0] * np.sqrt(5))
print("explained-variance ratios:", evr)

In [ ]:
# Self-check for task T1.2: run it; it should print OK.
assert lam[0] >= lam[1], "eigenvalues must be in descending order"
assert np.allclose(S @ V, V * lam), "columns of V must be eigenvectors of S"
assert np.allclose(Z.var(axis=0, ddof=1), lam), "variance of each score column must equal its eigenvalue"
assert np.allclose(np.corrcoef(Z, rowvar=False)[0, 1], 0), "scores must be uncorrelated"
assert np.isclose(evr.sum(), 1)
print("OK")

**Q1.** Compare `V[:, 0]` with the hand answer $v_1=(2,1)^\top/\sqrt5$ from the lecture. Explain any difference, and say whether it changes
(a) the explained variance, (b) the scores, and (c) the interpretation of PC1. (3–5 sentences)

*Your answer:* 

## Part 2 — PCA through the SVD, against scikit-learn (about 40 minutes)

Software computes PCA from the SVD of the centred data, $X_c=U\Sigma V^\top$. Then $\lambda_j=\sigma_j^2/(n-1)$, the components are the rows of $V^\top$, and the scores are $U\Sigma$.
We use the **Wine** data: 178 wines, 13 chemical measurements, 3 cultivars (UCI 109, CC BY 4.0).

In [ ]:
Xw, yw = tsds.data.load("wine")
tsds.data.info("wine")
Zw = StandardScaler().fit_transform(Xw)      # standardised: PCA of Zw is correlation PCA
print(Xw.shape)
Xw.head()

In [ ]:
# Task T2
def pca_svd(X, k):
    """PCA of the rows of X via the SVD of the centred data.

    Returns a dict with
      components         : (k, p) array, rows = unit principal directions (like sklearn's components_)
      explained_variance : (k,) array, eigenvalues lambda_1..lambda_k  (divisor n - 1)
      scores             : (n, k) array, coordinates of each row on the components
      mean               : (p,) array, the column means used for centring
    """
    X = np.asarray(X, dtype=float)
    # YOUR CODE HERE
    raise NotImplementedError()

mine = pca_svd(Zw, 3)
mine["explained_variance"]

In [ ]:
# Self-check for task T2: run it; it should print OK.
sk = PCA(n_components=3).fit(Zw)
assert np.allclose(mine["explained_variance"], sk.explained_variance_), "eigenvalues differ from sklearn"
signs = np.sign(np.sum(mine["components"] * sk.components_, axis=1))   # align arbitrary signs
assert np.allclose(mine["components"] * signs[:, None], sk.components_, atol=1e-8), "directions differ (after sign alignment)"
assert np.allclose(mine["scores"] * signs, sk.transform(Zw), atol=1e-8), "scores differ (after sign alignment)"
raw = pca_svd(Xw, 2)                       # raw Wine is NOT centred: your function must centre it
assert np.allclose(raw["explained_variance"], PCA(2).fit(Xw).explained_variance_),     "wrong on uncentred data: did you subtract the column means before the SVD?"
print("OK — sign flips needed per component:", signs)

## Part 3 — Reconstruction error equals the discarded variance (about 30 minutes)

Lecture §5.4 states that the rank-$k$ reconstruction $\hat x_i = \bar x + V_kV_k^\top(x_i-\bar x)$ satisfies
$\frac{1}{n-1}\sum_i\|x_i-\hat x_i\|^2=\sum_{j>k}\lambda_j$. Verify it on standardised Wine for every $k = 0, 1, \dots, 13$.

In [ ]:
# Task T3
# Fill two arrays of length 14 (k = 0..13):
#   err[k]  = sum of squared reconstruction residuals / (n - 1), using the first k components
#   tail[k] = sum of the eigenvalues lambda_{k+1} + ... + lambda_13
# You may use your pca_svd function. For k = 0, the "reconstruction" is just the mean.
p = Zw.shape[1]
n = Zw.shape[0]
# YOUR CODE HERE
raise NotImplementedError()
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(p + 1), err, "o-", label="reconstruction error / (n-1)")
ax.plot(range(p + 1), tail, "x--", label="sum of discarded eigenvalues")
ax.set_xlabel("number of components k"); ax.set_ylabel("variance units"); ax.legend(); plt.show()

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert err.shape == (14,) and tail.shape == (14,)
assert np.allclose(err, tail), "the identity should hold exactly (up to rounding)"
assert np.isclose(err[0], 13 * n / (n - 1)) or np.isclose(err[0], Zw.var(axis=0, ddof=1).sum())
assert np.all(np.diff(err) <= 1e-10), "error must never increase with k"
print("OK")

## Part 4 — Covariance PCA or correlation PCA? (about 25 minutes)

Look at the scale of the raw variables first.

In [ ]:
Xw.var().sort_values(ascending=False).round(2).head(6)

In [ ]:
# Task T4
# (a) evr_cov: explained-variance ratios of covariance PCA (PCA on the raw Xw)
# (b) evr_cor: explained-variance ratios of correlation PCA (PCA on the standardised Zw)
# (c) loadings_cor: a DataFrame (13 rows = variables, columns "PC1", "PC2") holding the
#     correlation-PCA component coefficients (sklearn's components_ transposed)
# YOUR CODE HERE
raise NotImplementedError()
print("covariance PCA EVR (first 3):", evr_cov[:3].round(4))
print("correlation PCA EVR (first 5):", evr_cor[:5].round(4))
loadings_cor.reindex(loadings_cor["PC1"].abs().sort_values(ascending=False).index).round(3)

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert np.isclose(evr_cov.sum(), 1) and np.isclose(evr_cor.sum(), 1)
assert loadings_cor.shape == (13, 2) and list(loadings_cor.columns) == ["PC1", "PC2"]
G = loadings_cor.to_numpy().T @ loadings_cor.to_numpy()
assert np.allclose(G, np.eye(2)), "loading columns must be orthonormal"
assert evr_cov[0] > evr_cor[0], "think about why covariance PCA concentrates variance on PC1"
print("OK")

**Q2.** (a) Which variable drives PC1 of **covariance** PCA, and why?
(b) For these data, which PCA is appropriate? Justify from the measurement units.
(c) Interpret PC1 and PC2 of **correlation** PCA in words, using the largest |coefficients| and their relative signs.
(About 120 words.)

*Your answer:* 

## Part 5 — Failure analysis: three ways PCA code goes wrong (about 20 minutes)

Run the three demonstrations, then answer Q3.

In [ ]:
# Failure 1: forgetting to centre. The SVD of the raw (uncentred) data finds the direction of the MEAN.
U, s, Vt = np.linalg.svd(Xw.to_numpy(), full_matrices=False)
m = Xw.to_numpy().mean(axis=0)
print("cosine(first right singular vector of UNcentred data, mean vector) =",
      round(abs(Vt[0] @ m) / np.linalg.norm(m), 5))

In [ ]:
# Failure 2: np.linalg.eig on a symmetric matrix. No ordering guarantee, and the dtype may be complex
# (this depends on the numpy version).
R = np.corrcoef(Zw, rowvar=False)
w_eig, _ = np.linalg.eig(R)
w_eigh, _ = np.linalg.eigh(R)
print("eig  dtype:", w_eig.dtype, "| first five:", np.round(w_eig[:5], 3))
print("eigh dtype:", w_eigh.dtype, "| first five (ascending):", np.round(w_eigh[:5], 3))

In [ ]:
# Failure 3: interpreting the overall sign. Two correct implementations, opposite signs.
v_sklearn = PCA(1).fit(Zw).components_[0]
v_svd = np.linalg.svd(Zw - Zw.mean(axis=0), full_matrices=False)[2][0]
print("sklearn PC1 (first 4):", np.round(v_sklearn[:4], 3))
print("raw SVD PC1 (first 4):", np.round(v_svd[:4], 3))

**Q3.** For each failure, state (i) what goes wrong, (ii) how you would detect it in your own work, and (iii) the fix.
Then state which of the three could change a *conclusion* of an analysis, rather than only how the numbers are displayed. (About 150 words.)

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *explain np.linalg.svd output shapes* | *used the explanation, wrote my own code* | *self-check T2 passed; compared with lecture §5.5* |
| | | | |

If you used no AI tool, write "None".